# P1. 계약과 샘플 결과 - 답을 **손으로 먼저 설계**해 보기

**왜 이걸 먼저 하나요?** 검색이나 LLM 코드를 짜기 전에 "**최종적으로 무엇을 돌려줘야 하는가**"를 정해야 합니다. 그것이 계약(`QueryOutput`)입니다. 이 노트북에서는 실제 문서의 쪽을 읽고, 질문 하나(Q07)에 대한 결과를 **내가 직접 설계**해서 최소 검사를 통과시킵니다. DB도 LLM도 필요 없습니다.

**진행 방식** - 저는(Claude) 힌트와 선택지를 드리고, 방향은 Codex 검토를 거친 의견으로 제안합니다. **최종 결정은 유종혁 님이 합니다.** 결정 칸(🧭)에 직접 적어 주세요. 의견은 **내 설계를 먼저 해 본 뒤에** 펼쳐 보세요.

**범위 (한 번에 다 하지 않습니다)**
- **Part A (이번 세션의 목표)** - 홍역 지침 **93쪽**(학교 담당교사의 역할)에서 주장 1~2개를 직접 설계하고 검증합니다.
- **Part B (이어서 / 도전)** - 의료기관 신고(홍역 지침 24쪽)와 법 조문(제11·12조)까지 넓혀 Q07 전체를 완성합니다.

**셀 표시 약속**
- 🟦 **제공** : 설계와 상관없는 보조 코드입니다. 그대로 실행하세요.
- ✍️ **직접** : 비어 있는 칸(`# TODO`)을 직접 채우는 셀입니다. 힌트를 먼저 읽으세요.
- 🔎 **생각해 보기** : 코드 없이 답을 먼저 떠올려 보는 질문입니다.
- 📦 **정답 보기** : 접혀 있습니다. **직접 해 본 뒤에** 펼치세요.
- 🧭 **결정** : 선택지와 의견을 보고 **내가 정하는** 칸입니다.

## 오늘의 질문 (Q07)

> 홍역 의심환자가 생겼을 때 학교 담당교사와 의료기관은 각각 보건소에 무엇을 신고하거나 안내하나요? 법률과 지침의 조건을 구분해 주세요.

이 질문이 어려운 이유: **"누가", "무엇을", "어떤 조건에서", "어떤 근거로"** 가 섞여 있고, 학교의 일 · 의료기관의 신고 · 법 조문이 **서로 다른 주체의 서로 다른 일**이라는 점입니다. 이것이 prevention_legal Graph의 핵심 과제입니다.

## 1. 🟦 준비

- 저장소 루트에서 `uv sync`를 한 환경의 `.venv` 커널을 선택하세요. 아래 `import`가 실패하면 먼저 **커널과 설치 환경**을 확인하세요.
- 이 연습의 `document_id`와 `chunk_id`는 **임시 형식**입니다. 실제 DB의 청크를 확인한 결과가 아닙니다.

In [ ]:
from rag.prevention_legal.contract_local import validate_query_output
from rag.prevention_legal.pages import DOCS, read_page, excerpt_in_page, make_citation

for key, (doc_id, filename) in DOCS.items():
    print(f"{key:8s} -> {doc_id}")

**이렇게 나오면 성공** - 문서 4개의 짧은 이름(`law`, `measles`, `hfmd`, `vaccine`)과 `document_id`가 나옵니다.

## 2. 🔎 계약을 쉬운 말로

| 이름 | 비유 | 한 줄 설명 |
|---|---|---|
| `QueryInput` | 업무 요청서 | 어떤 질문을, 어떤 문서 범위에서 찾을지 |
| `Finding` | 확인한 사실 카드 | **하나의** 주장 (그 주장에 쓴 근거 청크 ID 목록 포함) |
| `Citation` | 증빙 위치표 | 그 주장의 원문이 **어느 문서 몇 쪽 어느 문장**인지 |
| `QueryOutput` | 결과 봉투 | 상태(`status`) + `findings` + `citations` + 경고 |

**핵심 규칙: Finding은 내용, Citation은 증거, QueryOutput은 봉투.**

🔎 **생각해 보기** - 왜 큰 답변 전체를 `Finding` 하나에 넣으면 안 될까요? (힌트: 근거가 3개 있을 때 *어느 근거가 어느 문장의* 것인지 알 수 있을까요?)

## 3. 🟦 원문 읽기

아래 셀은 해당 쪽의 원문을 보여 줍니다. **가독성을 위해 빈 줄 · 이미지 링크 · 구분선을 뺀** 텍스트이며, 원문 그대로가 아닙니다. 표를 글로 풀어서 줄바꿈이 많고 `**`, `[*]` 같은 기호가 섞여 있습니다. **인용 후보를 찾은 뒤에는 원본 PDF에서 표의 제목 · 행 · 열과 주체를 직접 확인하세요.**

### Part A - 홍역 지침 93쪽
읽으면서 메모하세요: **누가**(주체), **어떤 조건에서**, **무엇을** 하라고 하나? (번호 항목마다 하나씩)

In [ ]:
print("===== 홍역 지침 93쪽 (부록 4-가: 초중고 학교 담당교사 역할) =====")
print(read_page("measles", 93)[:1600])

### Part B - 의료기관 신고와 법 조문 (이어서 할 때 읽으세요)
1. 의료기관은 언제까지 누구에게 신고하나? (홍역 지침 **24쪽** 표)
2. 법은 학교의 관리인 · 대표자에게도 신고 의무를 두나? 그 조건은? (감염병예방법 **제12조, 19쪽**)
3. 법 **제11조(18쪽)** 는 의사 · 의료기관장의 보고와 신고를 정합니다. ⚠️ **이 법 파일에는 제11조가 두 번 나옵니다.** 한 쪽에 `[시행일: 2027. 10. 21.]` 표시가 붙어 있습니다. **현행 조문과 시행 예정 조문이 함께 들어 있을 수 있으니, 인용하기 전에 어느 것이 지금 시행 중인지 시행일 표시로 직접 확인하세요.**

In [ ]:
print("===== 홍역 지침 24쪽 (발생 신고·보고) =====")
print(read_page("measles", 24)[:1100])

In [ ]:
print("===== 감염병예방법 19쪽 (제12조 그 밖의 신고의무자) =====")
t = read_page("law", 19)
print(t[t.find("제12조"):][:1100])

## 4. ✍️ 설계 ① - 요청서(`QueryInput`) 채우기

**해야 할 일** - 아래 `my_query`의 빈 칸을 채우세요.

**힌트**
- `region`: `"NATIONAL"`(전국 공통)과 `"GYEONGGI"`(경기도) 중 무엇일까요? Q07의 근거 문서가 **전국 공통 지침과 법률**이라는 점을 떠올려 보세요.
- `allowed_document_ids`: **호출한 쪽이 접근을 허용한 문서의 `document_id` 목록**입니다. 🔎 **빈 목록 `[]`은 "전체 허용"이 아니라 "허용된 문서 없음"** 입니다. 위 `DOCS`의 값을 쓰세요.
- 이 연습에서는 범위를 **홍역 지침과 법률 두 문서**로 일부러 제한해 봅니다. 실제 호출에서는 허용 범위를 담당 문서나 예상 정답 문서로 임의로 좁히지 않습니다. (모든 Graph는 공통 DB의 허용된 전체 문서를 볼 수 있습니다.)
- `preferred_document_ids`(선택)는 **우선순위 힌트**이고 `allowed`의 **부분집합**이어야 합니다.

In [ ]:
my_query = {
    "question": "홍역 의심환자가 생겼을 때 학교 담당교사와 의료기관은 각각 보건소에 무엇을 신고하거나 안내하나요? 법률과 지침의 조건을 구분해 주세요.",
    "region": ...,                 # TODO: "NATIONAL" 또는 "GYEONGGI"
    "allowed_document_ids": [],    # TODO: document_id 문자열들 (DOCS 의 값)
    # "preferred_document_ids": [],  # (선택) allowed 의 부분집합
}

In [ ]:
# 🟦 제공: 입력이 계약에 맞는지 점검 (최소 출력 검사는 입력을 검사하지 않으므로 따로 봅니다)
def check_input(q):
    problems = []
    if q.get("region") not in ("NATIONAL", "GYEONGGI"):
        problems.append("region 이 비어 있거나 올바른 값이 아닙니다")
    allowed = q.get("allowed_document_ids") or []
    if not allowed:
        problems.append("allowed_document_ids 가 비어 있습니다 (= 허용된 문서 없음)")
    if not set(q.get("preferred_document_ids", [])) <= set(allowed):
        problems.append("preferred_document_ids 가 allowed 의 부분집합이 아닙니다")
    print("입력 점검:", problems if problems else "문제 없음")

check_input(my_query)

<details>
<summary>📦 정답 보기</summary>

```python
my_query["region"] = "NATIONAL"
my_query["allowed_document_ids"] = [DOCS["measles"][0], DOCS["law"][0]]
```
`vaccine`, `hfmd`를 더 넣어도 틀린 것은 아닙니다. 다만 이 질문이 필요로 하지 않는 문서를 넓히면 검색 범위가 불필요하게 커집니다.

</details>

## 5. ✍️ 설계 ② - 빈 카드에 **주체 · 행동 · 조건 · 출처** 적기 (Part A: 93쪽)

**해야 할 일** - 93쪽에서 **카드를 1~2장** 만드세요. 카드 한 장 = **한 주체가 하는 한 가지 행동**입니다. 아직 몇 개의 `Finding`으로 나눌지는 정하지 않습니다. 먼저 카드를 채우세요.

**힌트 (카드 칸마다 이렇게 생각해 보세요)**
- **주체**: 이 문장의 "누가"는? 문장에 직접 있나요, 페이지 제목에만 있나요? (93쪽 맨 위 제목을 다시 보세요.)
- **행동**: 한 문장에 행동이 **두 개** 들어 있지는 않나요? (예: "~하도록 안내하고, ~에 신고") 둘은 같은 행동일까요?
- **조건**: 어떤 상황에서 적용되나요? 원문에 있는 조건어를 빠뜨리지 마세요. ("진단을 받은 경우"인지 "받지 않았으나"인지 같은 한 구절이 결과를 바꿉니다.)
- **출처**: 문서 키(`measles`), PDF 쪽, 그리고 **원문 한 구간**.

In [ ]:
my_cards = [
    # 한 장 예시 모양 (내용은 직접 채우세요):
    # {"주체": "...", "행동": "...", "조건": "...", "출처": ("measles", 93, "원문에서 그대로 복사한 구간")},
]

## 6. 🧭 설계 ③ - 카드를 **몇 개의 `Finding`으로 묶을까?** (결정)

> 위 카드를 먼저 채운 뒤에 아래 의견을 펼쳐 보세요.

원칙: **`Finding` 하나 = 주장 하나(원자적)**. 안내 · 신고 · 보고처럼 **법적 의미가 다른 행동은 따로** 둡니다.

🔎 **생각해 보기**
- 한 문장에 행동이 둘 들어 있을 때(안내 + 신고), 같은 원문 근거를 **두 `Finding`이 함께 써도** 될까요? (같은 청크는 여러 `Finding`이 재사용할 수 있습니다.)
- 법 제12조의 "관리인이 신고한다"와 지침의 "학교 담당교사가 보건소에 신고한다"를 **같은 말**로 합쳐도 될까요? 출처와 의미가 같은지 생각해 보세요.

<details>
<summary>📦 방향 제시 - Claude · Codex 의견 (내 카드를 먼저 채운 뒤에 펼치세요)</summary>

**선택지**
| 안 | 쪼개는 기준 | 장점 | 단점 |
|---|---|---|---|
| 큰 단위 | 주체별로 묶기 (학교 / 의료기관 / 법) | 읽기 쉽다 | 서로 다른 행동(안내 vs 신고)이 한 주장에 섞인다 |
| 행동 단위 | 주체 + 행동 + 조건마다 따로 | 근거와 검증이 1:1로 맞는다 | 개수가 많고 쪼개는 기준이 필요하다 |

**Claude 의견** - 행동 단위 쪽을 권합니다. 이 Graph의 평가 기준이 "의료기관 신고를 학교 행동으로 잘못 연결하지 않는가"여서, 주체와 행동이 다르면 `Finding`도 따로 있어야 나중에 검증하기 쉽습니다.

**Codex 의견** - 행동 단위에 가까운 분할을 권합니다. 먼저 주체 · 행동 · 조건 · 근거를 적고, 안내와 신고처럼 의미가 다른 행동은 분리해 보세요. 출발점은 학교 안내 / 학교 신고 / 의료기관 신고 / 법 제11조 / 법 제12조지만 **개수를 정답으로 고정하지 않습니다.** 각 주장에 연결한 발췌가 주체와 조건까지 뒷받침하는지 확인한 뒤 최종 분할은 직접 결정하세요.

</details>

🧭 **내 결정** (큰 단위 / 행동 단위 / 내 방식, 그리고 이유 한 줄)

> (여기에 적기)

## 7. ✍️ 설계 ④ - 주장마다 **근거 문장**(여러 개 가능) 연결하기

**해야 할 일** - 정한 `Finding`마다 **주장 한 문장**과 **그 주장을 뒷받침하는 근거 행(하나 이상)** 을 `my_findings`에 적으세요. 그리고 확인 셀로 발췌문이 원문에 있는지 보세요.

**힌트**
- 한 주장에 근거가 **여러 개** 필요할 수 있습니다. (예: 행동은 번호 항목에, 주체는 페이지 제목에 있을 때) 그때는 `evidence`에 행을 여러 개 넣으세요. 같은 근거를 다른 주장이 또 써도 됩니다.
- 발췌문은 **원문에서 그대로** 가져옵니다. 직접 쓴 문장이나 의역은 안 됩니다. 🔎 왜일까요? (나중에 "근거가 정말 있는가"를 코드로 확인하려면?)
- 확인 함수는 공백 · 줄바꿈 · `**` · `[ ]`를 무시하는 **간단한 후보 확인**입니다. `확인`이 나와도 표의 다른 줄이 이어졌을 수 있으니 **원문 문맥을 직접 보세요**.
- 같은 쪽의 서로 다른 문장을 별개 근거로 쓰려면 `block` 번호를 다르게 줍니다(`b01`, `b02`). 같은 `block`이면 같은 `chunk_id`가 되고, **같은 `chunk_id`의 `Citation`은 결과에 한 번만** 담아야 합니다.

In [ ]:
my_findings = [
    # 한 주장의 모양 (내용은 직접 채우세요):
    # {
    #     "claim": "...주장 한 문장...",
    #     "evidence": [
    #         {"doc": "measles", "page": 93, "block": 1, "excerpt": "원문에서 그대로 복사한 구간"},
    #         # 근거가 더 필요하면 행을 더 추가
    #     ],
    # },
]

In [ ]:
# 🟦 제공: 내가 적은 발췌문이 원문에 있는지 후보 확인
for i, f in enumerate(my_findings, 1):
    for ev in f["evidence"]:
        ok = excerpt_in_page(ev["excerpt"], ev["doc"], ev["page"])
        print(f"[주장 {i}] {ev['doc']} {ev['page']}쪽 b{ev['block']:02d}: "
              f"{'후보 확인' if ok else '못 찾음 - 기호나 글자를 확인하세요'}")
if not my_findings:
    print("아직 my_findings 가 비어 있습니다. (근거를 아직 연결하지 않은 상태입니다)")

**이렇게 나오면 성공** - 모든 줄이 `후보 확인`. `못 찾음`이면 위 3번 출력과 한 글자씩 비교해 보세요.

<details>
<summary>📦 막힐 때 힌트</summary>

- 93쪽 **번호 1번 · 3번 항목** 문장과 **맨 위 제목**(주체가 담긴 곳)을 함께 보세요.
- 항목 문장 중간에 `[*]`나 `**`가 끼어 있으면, 그 앞이나 뒤의 짧은 구간만 발췌해도 됩니다.

</details>

## 8. 🧭 설계 ⑤ - `status`와 `warnings` (결정)

**상태 선택지**

| 값 | 뜻 | 이 질문에서 고를 때 |
|---|---|---|
| `ANSWERED` | 맡은 범위에서 충분한 근거가 있음 | 필요한 사실과 적용 조건이 모두 원문으로 확인됐을 때 |
| `PARTIAL` | 일부만 확인됨. 빠진 부분은 `warnings`에 | 적용 조건을 원문으로 다 확인하지 못했을 때 |

🔎 **생각해 보기**
- 지침의 "학교 담당교사가 보건소에 신고"는 **지침의 안내**입니다. 이것을 "법적 의무"라고 말해도 될까요?
- (Part B를 한다면) 법 제12조는 "**보건복지부령으로 정하는 감염병**"이라고 합니다. 이 원문만으로 "홍역에서 학교가 반드시 신고해야 한다"고 말해도 될까요?
- 확인하지 못한 내용은 **`Finding`에서 빼고, 빠진 내용과 이유를 `warnings`에 적는 것**을 함께 할 수 있습니다. 둘은 양자택일이 아닙니다.
- 그리고 **이번 답이 어디까지를 다룬 것인지**(Part A만 했다면 의료기관·법 부분은 다루지 않았다는 사실)도 `status`와 `warnings`에 어떻게 드러낼지 생각해 보세요.

<details>
<summary>📦 방향 제시 - Claude · Codex 의견 (내 생각을 먼저 정한 뒤에 펼치세요)</summary>

**Claude 의견** - 확인된 사실은 `Finding`으로 두고, "적용 여부는 원문(예: 복지부령) 확인이 필요하다"는 한계는 `warnings`에 적는 방식을 권합니다. 법 적용 조건까지 닫지 못했다면 `PARTIAL`이 더 정직합니다.

**Codex 의견** - 복지부령상 홍역 적용 여부까지 답하려면 `PARTIAL`을 권합니다. 경고 예: "법 제12조의 홍역 적용 여부는 관련 보건복지부령을 확인하지 못했습니다." 질문의 범위를 법의 일반 조건 설명까지로 해석하면 `ANSWERED`도 가능합니다. 그래서 골든 문서의 `ANSWERED` 기대값과 **판단 기준을 함께 정리**해야 합니다.

</details>

🧭 **내 결정** (`status` 값과 `warnings` 문구, 이유)

> (여기에 적기)

## 9. ✍️ 조립 - `QueryOutput` 만들기

**해야 할 일** - 위 결정들로 `my_output`을 완성하세요. 필드 일곱 개가 모두 있어야 합니다.

**힌트**
- `domain`은 우리 Graph의 이름 `"prevention_legal"`로 고정입니다.
- `Citation`은 직접 쓰지 말고 `make_citation(문서키, 쪽, 발췌, block=번호)`를 쓰세요. `document_id`와 `chunk_id`가 형식에 맞게 만들어지고, 발췌문이 원문에 없으면 `ValueError`로 알려 줍니다.
- `my_findings`를 **하나씩 돌며** 각 `evidence` 행마다 `make_citation`을 만들고, **이미 만든 `chunk_id`는 다시 담지 않습니다.** 그리고 `Finding`의 `chunk_ids`에는 그 주장의 근거 `chunk_id`를 모두 넣습니다.
- `Finding`의 `chunk_ids`는 **같은 결과의 `citations` 안에 있는 `chunk_id`** 여야 합니다.
- 아직 `SchoolAction` 노드를 DB에 만들지 않았으므로 `action_ids`, `disease_ids`, `concept_ids`는 `[]`로 두어도 됩니다. 최상위 `action_ids`는 `Finding`들의 `action_ids` 합집합(지금은 `[]`)입니다.
- `answer`는 단독 시연용 요약이라 `""`도 허용됩니다.
- 🔎 `make_citation`이 `ValueError`를 내면 **어느 근거 행이 문제인지** 어떻게 알 수 있을까요? (힌트: 돌리는 도중 어떤 행인지 출력해 보세요.)

In [ ]:
citations = []
findings = []

# TODO 1: my_findings 를 돌며 make_citation(...) 으로 Citation 을 만들어 citations 에 넣기 (같은 chunk_id 는 한 번만)
# TODO 2: 각 주장을 {"claim": ..., "chunk_ids": [...], "action_ids": [], "disease_ids": [], "concept_ids": []} 로 만들어 findings 에 넣기
# TODO 3: 아래 my_output 의 status 와 warnings 를 8번에서 정한 대로 채우기

my_output = {
    "domain": "prevention_legal",
    "status": ...,       # TODO
    "findings": findings,
    "citations": citations,
    "action_ids": [],
    "warnings": [],      # TODO
    "answer": "",
}

## 10. 🟦 검증 - 내 설계를 점검하기

In [ ]:
check_input(my_query)

try:
    validate_query_output(my_output, my_query)
    print("최소 출력 검사 통과")
except Exception as e:
    print("최소 출력 검사 실패:", type(e).__name__, "-", e)

# 발췌문 후보 확인 (잘못된 document_id 가 있어도 멈추지 않는다)
key_of = {doc_id: key for key, (doc_id, _) in DOCS.items()}
for cit in my_output["citations"]:
    key = key_of.get(cit["document_id"])
    if key is None:
        print(f"{cit['chunk_id']:44s} 알 수 없는 document_id")
        continue
    ok = excerpt_in_page(cit["excerpt"], key, cit["pdf_page"])
    print(f"{cit['chunk_id']:44s} 발췌문 {'후보 확인' if ok else '못 찾음'}")
print("Finding 수:", len(my_output["findings"]), "| Citation 수:", len(my_output["citations"]))

**이렇게 나오면 성공** - `입력 점검: 문제 없음`, `최소 출력 검사 통과`, 모든 발췌문 `후보 확인`.

⚠️ **"최소 출력 검사 통과"의 뜻** - 필수 출력 구조의 일부를 확인했다는 뜻입니다. **입력의 지역 · 허용 범위, 실제 DB 청크 존재, 발췌의 표 문맥, 주장의 사실성은 이 검사가 확인하지 않습니다.** (그래서 위의 입력 점검과 원문 직접 대조가 필요합니다.)

**자주 나오는 오류와 뜻**
| 메시지 | 뜻 | 확인할 것 |
|---|---|---|
| `Missing keys` | 필수 키가 빠짐 | 7개 키가 다 있나? |
| `Unknown status` | `status` 값이 올바르지 않음 | 아직 `...`로 남아 있나? |
| `Citation outside allowed documents` | 허용 안 한 문서를 인용 | 4번의 `allowed_document_ids` |
| `Duplicate citation chunk_id` | 같은 `chunk_id`의 Citation을 두 번 담음 | 같은 문서 · 쪽 · `block` 조합으로 만든 Citation이 중복됐나? |
| `Finding cites a missing chunk` | `chunk_ids`가 `citations`에 없음 | 철자 · 형식 |
| `Top-level action_ids must match findings` | 최상위 `action_ids` ≠ `Finding`들의 합집합 | 둘 다 `[]`면 통과 |
| `Answered/partial result requires a finding` | 근거 없는데 ANSWERED/PARTIAL | `findings`가 비었나? |

## 11. 📦 참고 답안과 비교 (직접 해 본 뒤에 펼치세요)

<details>
<summary>📦 Claude의 참고 답안 (행동 단위, status=PARTIAL) - Part A + Part B 일부</summary>

이 답안은 **하나의 예시**이며 정답이 아닙니다. 법 제11조는 이 파일에 현행과 시행 예정 조문이 함께 있어 **일부러 뺐습니다**(Part B 도전 과제).

```python
# 93쪽: 번호 항목 b02/b03, 제목(주체) b01
title = make_citation("measles", 93, "초 중 고등학교 내 환자 발생 시 학교[담당교사] 역할", block=1)
item1 = make_citation("measles", 93, "교내 홍역(의심)환자 발생 시 반드시 의료기관에서 진료 받도록 안내하고, 관할 보건소에 신고", block=2)
item3 = make_citation("measles", 93, "홍역(의심) 진단을 받지 않았으나 유사증상자가 2주 이내 2명 이상 발생 할 경우 보건소로 신고", block=3)
# 24쪽 표(풀린 글): 의료기관 신고
med = make_citation("measles", 24, "(의사)환자를 진단한 경우 발생 24시간 이내에 관할보건소로 신고", block=1)
# 법 제12조 (19쪽): 조건 문장 b01, 2호(학교 관리인) b02
law_cond = make_citation("law", 19, "제1급감염병부터 제3급감염병까지에 해당하는 감염병 중 보건복지부령으로 정하는 감염병이 발생한 경우에는 의사, 치과의사 또는 한의사의 진단이나 검안을 요구하거나 해당 주소지를 관할하는 보건소장에게 신고하여야 한다", block=1)
law_who = make_citation("law", 19, "학교, 사회복지시설, 병원, 관공서, 회사, 공연장, 예배장소, 선박ㆍ항공기ㆍ열차 등 운송수단, 각종 사무소ㆍ사업소, 음식점, 숙박업소 또는 그 밖에 여러 사람이 모이는 장소로서 보건복지부령으로 정하는 장소의 관리인, 경영자 또는 대표자", block=2)

def F(claim, *cits):
    return {"claim": claim, "chunk_ids": [c["chunk_id"] for c in cits],
            "action_ids": [], "disease_ids": [], "concept_ids": []}

my_output = {
    "domain": "prevention_legal",
    "status": "PARTIAL",
    "findings": [
        F("지침: 학교 담당교사는 교내에 홍역(의심)환자가 발생하면 의료기관에서 진료 받도록 안내한다.", title, item1),
        F("지침: 학교 담당교사는 교내에 홍역(의심)환자가 발생하면 관할 보건소에 신고한다.", title, item1),
        F("지침: 홍역(의심) 진단을 받지 않았더라도 유사증상자가 2주 이내 2명 이상 발생하면 학교 담당교사는 보건소로 신고한다.", title, item3),
        F("지침: 의료기관은 (의사)환자를 진단한 경우 24시간 이내에 관할 보건소로 신고한다.", med),
        F("법 제12조: 제1~3급감염병 중 보건복지부령으로 정하는 감염병이 발생한 경우, 학교 등 여러 사람이 모이는 장소의 관리인·경영자·대표자는 의사 등의 진단·검안을 요구하거나 관할 보건소장에게 신고하여야 한다.", law_cond, law_who),
    ],
    "citations": [title, item1, item3, med, law_cond, law_who],
    "action_ids": [],
    "warnings": [
        "법 제12조의 신고 대상 감염병은 보건복지부령으로 정하므로, 홍역에 학교 관리인의 신고 의무가 적용되는지는 복지부령 원문 확인이 필요합니다.",
        "지침의 '학교 담당교사 신고' 안내와 법 제12조의 신고의무를 같은 것으로 단정하지 않았습니다.",
        "24쪽 표는 글로 풀려 있어 '의료기관' 열과 신고 행의 대응은 원본 PDF 표에서 확인이 필요합니다.",
        "법 제11조(의사·의료기관장의 보고·신고)는 현행과 시행 예정(2027-10-21) 조문이 함께 있어 이번 답에 포함하지 않았습니다.",
    ],
    "answer": "",
}
```

**비교해 볼 점**
- 안내와 신고를 따로 두고, **같은 근거 청크(`item1`)를 두 주장이 함께** 쓴 것을 어떻게 생각하나요?
- 주체(`title`)를 별도 근거로 연결한 이유는? 항목 문장에는 "담당교사"가 직접 없습니다.
- 법 제12조 주장이 근거 청크 2개(조건 + 주체)를 쓴 이유는?
- 내 `status`와 `warnings`는 어떻게 달랐고, 왜 그렇게 정했나요?
- 이 참고 답안이 **과하게 단정했거나 틀린 곳**은 없나요? (원문과 직접 대조해 보세요. 예: 24쪽 표의 '의료기관' 열.)

</details>

## 12. 도전 (이어서) - Part B를 넓혀 Q07 완성하기

- [ ] 의료기관 신고(24쪽)를 직접 카드로 만들기
- [ ] 법 제12조(19쪽)의 **조건**과 **주체**를 나눠 근거 연결하기
- [ ] 법 **제11조(18쪽)**: 두 벌의 조문 중 **현재 시행 중인 쪽**을 `[시행일: 2027. 10. 21.]` 표시로 찾아내기. 어느 것을 인용할지, 어떤 경고를 달지 정하기
- [ ] (생각) 지침의 신고 안내 vs 법의 신고의무 - 한 `Finding`으로 합쳐도 되는가?

## 13. 정리와 다음 단계

- [ ] 4번: `my_query` 완성, `입력 점검: 문제 없음`
- [ ] 5번 · 6번: 카드와 🧭 결정을 적었다
- [ ] 7번: 모든 발췌문 `후보 확인` (원문 문맥도 직접 봤다)
- [ ] 8번: 🧭 `status` · `warnings` 결정을 적었다
- [ ] 10번: `최소 출력 검사 통과`
- [ ] 11번: 참고 답안과 비교하고 차이를 한 줄 적었다

🧭 **오늘 배운 것 / 헷갈린 것** (한두 줄)

> (여기에 적기)

**다음: P2 문서 목록(`source_manifest.csv`)** - 이번 노트북의 `document_id` 임시 값을 정식으로 정하고, 문서별 판본 · 발행기관 · 이용 조건을 기록합니다.